In [3]:
# import modules
import numpy as np
import pandas as pd
from typing import Set, List, Dict

# make sure its using colab to connect with google drive for storing output?
import os
import re
import ast
from google.colab import drive

In [ ]:
# .csv data of the filtered assessment


In [4]:
# function to extract the fragment sequence
def extract_fragment_id(filename: str) -> str:
    """
    Extract fragment identifier from filename.
    Expects a 6-digit number where first3=start, last3=end.
    Example: 'LepR_602608.csv' -> '602-608'
    """
    match = re.search(r'(\d{6})', filename)
    if not match:
        raise ValueError(f"Filename {filename} does not contain a 6-digit number")
    digits = match.group(1)
    start, end = digits[:3], digits[3:]
    return f"{start}-{end}"

In [5]:
# function to get the LepR residue position of each identified interaction series of a fragment
def get_position_set(row: pd.Series) -> Set[int]:
    """
    Extract the set of residue positions from a row.
    Uses the 'positions' column if available (as list), otherwise generates
    from 'start_position' to 'end_position' inclusive.
    """
    if 'positions' in row and pd.notna(row['positions']):
        # Convert string representation of list to actual list
        try:
            pos_list = ast.literal_eval(row['positions'])
            return set(pos_list)
        except:
            # Fallback: treat as comma-separated?
            pass
    # Fallback to start/end range
    start = int(row['start_position'])
    end = int(row['end_position'])
    return set(range(start, end + 1))

In [ ]:
# function to compare a specific fragment with the original leptin interaction
def compare_fragments(original_csv: str, test_folder: str, output_csv: str = None):
    """
    Compare original leptin file against all test fragment CSVs.
    
    Parameters:
    - original_csv: path to the original leptin CSV file
    - test_folder: directory containing fragment test CSV files
    - output_csv: optional output path; if None, prints dataframe
    """
    # Read original dataframe
    df_original = pd.read_csv(original_csv)
    
    # For each row, compute its position set
    original_sets = []
    for idx, row in df_original.iterrows():
        original_sets.append(get_position_set(row))
    
    # Initialize list for each row to store overlapping fragment IDs
    overlapping_fragments: Dict[int, List[str]] = {i: [] for i in range(len(df_original))}
    
    # Process each test file in the folder
    for filename in os.listdir(test_folder):
        if not filename.lower().endswith('.csv'):
            continue
        
        filepath = os.path.join(test_folder, filename)
        try:
            fragment_id = extract_fragment_id(filename)
        except ValueError as e:
            print(f"Skipping {filename}: {e}")
            continue
        
        # Read test CSV
        df_test = pd.read_csv(filepath)
        
        # Collect all positions from all rows of this test file
        test_positions: Set[int] = set()
        for _, row in df_test.iterrows():
            test_positions.update(get_position_set(row))
        
        if not test_positions:
            continue
        
        # Check overlap with each original row
        for idx, orig_positions in enumerate(original_sets):
            if orig_positions & test_positions:  # intersection not empty
                overlapping_fragments[idx].append(fragment_id)
    
    # Add column to original dataframe
    df_original['overlapping_fragments'] = [
        '; '.join(frag_ids) if frag_ids else ''
        for frag_ids in overlapping_fragments.values()
    ]
    
    # Output
    if output_csv:
        df_original.to_csv(output_csv, index=False)
        print(f"Saved to {output_csv}")
    else:
        print(df_original.to_string())
    
    return df_original
